In [20]:
import requests
from bs4 import BeautifulSoup
from supabase import create_client
import os
import json

SUPABSE_URL = os.getenv("SUPABASE_URL")
SUPABSE_KEY = os.getenv("SUPABASE_KEY")
BASE_URL = "https://asia.pokemon-card.com/ph/event-search/"
CACHE_FILE = 'results.json'

supabase = create_client(SUPABSE_URL, SUPABSE_KEY)
session = requests.Session()

headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36'
}

def get_events_with_pending_results() -> list[int]:
  try:
    response = supabase.table("events").select("id").order("date").execute()
    
    existing_results = supabase.table("results").select("event_id").eq("is_temp", False).execute()
    
    event_ids_with_results = {r["event_id"] for r in existing_results.data}
    event_ids_without_results = [e["id"] for e in response.data if e['id'] not in event_ids_with_results]
    
    print(f"🔍 Found {len(event_ids_without_results)} events pending result scraping.")
    
    return event_ids_without_results
  except Exception as e:
    print(f"❌ Error fetching pending events: {e}")
    return []

def scrape_updated_event_data(event_id: int, soup: BeautifulSoup):
  venue_el = soup.select("#place div")
  venue = venue_el[-1].get_text(strip=True)
  
  supabase.table("events").update({ "venue": venue }).eq("id", event_id).execute()

def scrape_event_results(event_id: int, soup: BeautifulSoup):
  rows = soup.select(".rankingTable .tableData")
  
  if not rows:
    print(f"No rows found for event ID {event_id}")
    return []

  results_data = []
  for row in rows:
    rank_el = row.select_one(".rank")
    rank = rank_el.get_text(strip=True)
    
    points_el = row.select_one(".point")
    points = int(points_el.get_text(strip=True).replace("pt", ""))
    
    trainer_id_el = row.select_one(".userId")
    trainer_id = trainer_id_el.get_text(strip=True)

    results_data.append({
      "event_id": event_id,
      "rank": rank,
      "trainer_id": trainer_id,
      "points": points
    })
  
  return results_data

all_results = []
if os.path.exists(CACHE_FILE):
  print(f"📂 {CACHE_FILE} found. Loading data from local cache instead of scraping...")
  with open(CACHE_FILE, 'r') as f:
    all_results = json.load(f)
else:
  for event_id in get_events_with_pending_results():
    res = session.get(f"{BASE_URL}{event_id}", headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')
    
    print(f"⏳ Scraping event ID {event_id}")
    
    scrape_updated_event_data(event_id, soup)
    results = scrape_event_results(event_id, soup)
    
    if results:
      all_results.extend(x for x in results if x not in all_results)

  with open('results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

unique_results = {}
for item in all_results:
  key = (item['event_id'], item['trainer_id'])
  unique_results[key] = item

final_payload = list(unique_results.values())


with open('results-deduped.json', 'w') as f:
  json.dump(final_payload, f, indent=2)

if final_payload:
  try:
    result = supabase.table("results").upsert(final_payload).execute()
    print(f"🚀 Successfully synced {len(final_payload)} results to Supabase.")
  except Exception as e:
    print(f"❌ Supabase Error: {e}")
    


📂 results.json found. Loading data from local cache instead of scraping...
🚀 Successfully synced 1369 results to Supabase.
